## SomeCens package tutorial 🌎📊

Package code in available at [SomeCens's github repository](github.com/jimenaRL/SoMeCens).   

#### In this tutorial we will use SomeCens package in order to:

- Create a Demograph tree object representing a country and its hierarchical organized geographic units (the country's geographical subdivisions) 🌐
- Load sociodemographic (age and gender) information for units 👶👧👵👦🧔👴
- Localize user in units 📌
- Create choropleth maps to show sociodemographic and localisation data 🗺️
- Export tables with localized user and units sociodemographic data 📊

We will use data already formatted in the correct way for using them with the Demograph class.
You can check the [scripts](https://github.com/jimenaRL/SoMeCens/tree/0fddfc2ff01611bf9f21af2a36b2e648b1e2bbd2/scripts) provided in the repository for formatting data for EU countries 🇪🇺, Chile 🇨🇱 and USA 🇺🇸.  

In [1]:
country = 'chile'
unitspath = "data/chile/chile_geounits_census_2024.csv"
agedistpath = "data/chile/chile_age_distribution_census_2024.csv"
genderdistpath = "data/chile/chile_gender_distribution_census_2024.csv"
usersdatapath = "data/chile/chile_metadata_2023.csv"

#### Load main package and configurations variables for some countries

In [2]:
import os, csv, time, yaml, json
from string import Template

In [3]:
from somecens import DemoGraph
from somecens.tools import matchUsersLocations, getCountryAliases, getUnitsAliases, getOtherCountriesNames

In [4]:
from somecens.chile.conf import CHILEAGECATS, CHILEGENDERCATS
from somecens.us.conf import USAGECATS, USGENDERCATS
from somecens.nuts.conf import NUTS3AGECATS, NUTS3GENDERCATS

#### 1. Demograph creation

In order to create a **Demograph object** representing a country (here Chile), we need to provide an iterable of dictionaries with the information of each geaographical units. The dictionaries must be of the form:

    {
        'code': 'FRJ24',
        'level': '3',
        'label': 'Gers',
        'parent_code': 'FRJ2'
    }

_There must be one and only one unit representing the highest country level with code "0" and a empty parent_code._

Here we load them from a flat csv file containing these informations.

In [5]:
!xan head data/chile/chile_geounits_census_2024.csv | xan v


Displaying 4 cols from 10 rows of <stdin>
┌───┬──────────────────────────────────┬──────┬───────┬─────────────┐
│ - │ label                            │ code │ level │ parent_code │
├───┼──────────────────────────────────┼──────┼───────┼─────────────┤
│ 0 │ Chile                            │    0 │     0 │     <empty> │
│ 1 │ Arica y Parinacota               │   15 │     1 │           0 │
│ 2 │ Tarapacá                         │    1 │     1 │           0 │
│ 3 │ Antofagasta                      │    2 │     1 │           0 │
│ 4 │ Atacama                          │    3 │     1 │           0 │
│ 5 │ Coquimbo                         │    4 │     1 │           0 │
│ 6 │ Valparaíso                       │    5 │     1 │           0 │
│ 7 │ Metropolitana de Santiago        │   13 │     1 │           0 │
│ 8 │ Libertador General Bernardo O'H… │    6 │     1 │           0 │
│ 9 │ Maule                            │    7 │     1 │           0 │
└───┴──────────────────────────────────┴──────┴

In [6]:
with open(unitspath, "r", encoding="utf-8") as f:
    geoUnits = [r for r in csv.DictReader(f)]

geoUnits[:3]

[{'label': 'Chile', 'code': '0', 'level': '0', 'parent_code': ''},
 {'label': 'Arica y Parinacota',
  'code': '15',
  'level': '1',
  'parent_code': '0'},
 {'label': 'Tarapacá', 'code': '1', 'level': '1', 'parent_code': '0'}]

We must also state which will be the age and gender categories of the country.

In [7]:
print(f"CHILE AGE CATEGORIES:\n\t{CHILEAGECATS}")

print(f"\nCHILE GENDER CATEGORIES:\n\t{CHILEGENDERCATS}")

CHILE AGE CATEGORIES:
	['Total', '0 a 4', '5 a 9', '10 a 14', '15 a 19', '20 a 24', '25 a 29', '30 a 34', '35 a 39', '40 a 44', '45 a 49', '50 a 54', '55 a 59', '60 a 64', '65 a 69', '70 a 74', '75 a 79', '80 a 84', '85 o más']

CHILE GENDER CATEGORIES:
	['female', 'male', 'total']


In [8]:
demo = DemoGraph(
    demography=geoUnits,
    genderCats=CHILEGENDERCATS, 
    ageCats=CHILEAGECATS
)

Created Chile (0) DemoGraph


We can show the tree structure of the demograph.

In [9]:
demo.showGeoUnits(max_level=1)

---------------------------------------------------------
GeoUnit
label: Chile
level: 0
code: 0
children: 15 | 1 | 2 | 3 | 4 | 5 | 13 | 6 | 7 | 16 | 8 | 9 | 14 | 10 | 11 | 12
gender distribution: None
age distribution: None
    ---------------------------------------------------------
    GeoUnit
    label: Arica y Parinacota
    level: 1
    code: 15
    children: p151 | p152
    gender distribution: None
    age distribution: None
    ---------------------------------------------------------
    GeoUnit
    label: Tarapacá
    level: 1
    code: 1
    children: p11 | p14
    gender distribution: None
    age distribution: None
    ---------------------------------------------------------
    GeoUnit
    label: Antofagasta
    level: 1
    code: 2
    children: p21 | p22 | p23
    gender distribution: None
    age distribution: None
    ---------------------------------------------------------
    GeoUnit
    label: Atacama
    level: 1
    code: 3
    children: p31 | p32 | p33
    ge

#### 2. Load sociodemographic data

We can now load to the Demograph age and gender data for the geophical units.

2.1 For _**gender distributions**_ we need to provide an iterable of dictionaries containing each the code of an unit and the values for each of the gender categories. They are of the form:

    {
        'code': 'FRJ24',
        'total': '18480432',
        'male': '8967033',
        'female': '9513399'
    }

we get them from a flat csv file containing these informations.

In [10]:
!xan head data/chile/chile_gender_distribution_census_2024.csv | xan v


Displaying 4 cols from 10 rows of <stdin>
┌───┬──────┬──────────┬─────────┬─────────┐
│ - │ code │ total    │ male    │ female  │
├───┼──────┼──────────┼─────────┼─────────┤
│ 0 │    0 │ 18480432 │ 8967033 │ 9513399 │
│ 1 │   15 │   244569 │  120381 │  124188 │
│ 2 │    1 │   369806 │  183343 │  186463 │
│ 3 │    2 │   635416 │  313995 │  321421 │
│ 4 │    3 │   299180 │  148263 │  150917 │
│ 5 │    4 │   832864 │  404397 │  428467 │
│ 6 │    5 │  1896053 │  913643 │  982410 │
│ 7 │   13 │  7400741 │ 3582833 │ 3817908 │
│ 8 │    6 │   987228 │  483948 │  503280 │
│ 9 │    7 │  1123008 │  545255 │  577753 │
└───┴──────┴──────────┴─────────┴─────────┘



In [11]:
with open(genderdistpath, "r") as f:
    genderDistribution = [r for r in csv.DictReader(f)]

genderDistribution[:3]

[{'code': '0', 'total': '18480432', 'male': '8967033', 'female': '9513399'},
 {'code': '15', 'total': '244569', 'male': '120381', 'female': '124188'},
 {'code': '1', 'total': '369806', 'male': '183343', 'female': '186463'}]

In [12]:
demo.setGenderDistributions(genderDistribution)

In [13]:
demo.showGeoUnits(max_level=0)

---------------------------------------------------------
GeoUnit
label: Chile
level: 0
code: 0
children: 15 | 1 | 2 | 3 | 4 | 5 | 13 | 6 | 7 | 16 | 8 | 9 | 14 | 10 | 11 | 12
gender distribution: {'female': '9513399', 'male': '8967033', 'total': '18480432'}
age distribution: None


2.2 For _**age distributions**_ we need to provide an iterable of dictionaries containing each the code of an unit and the value for each one of the age categories. They are of the form:


In [14]:
!xan head -l 25 data/chile/chile_age_distribution_census_2024.csv | xan v


Displaying 3 cols from 25 rows of <stdin>
┌────┬──────┬──────────┬──────────┐
│ -  │ code │ age      │ total    │
├────┼──────┼──────────┼──────────┤
│ 0  │    0 │ 10 a 14  │  1256440 │
│ 1  │    0 │ 15 a 19  │  1219347 │
│ 2  │    0 │ 20 a 24  │  1273193 │
│ 3  │    0 │ 25 a 29  │  1383669 │
│ 4  │    0 │ 30 a 34  │  1527489 │
│ 5  │    0 │ 35 a 39  │  1408198 │
│ 6  │    0 │ 40 a 44  │  1270493 │
│ 7  │    0 │ 45 a 49  │  1151776 │
│ 8  │    0 │ 50 a 54  │  1173352 │
│ 9  │    0 │ 55 a 59  │  1133239 │
│ 10 │    0 │ 60 a 64  │  1077790 │
│ 11 │    0 │ 65 a 69  │   870801 │
│ 12 │    0 │ 70 a 74  │   646241 │
│ 13 │    0 │ 75 a 79  │   477186 │
│ 14 │    0 │ 80 a 84  │   317424 │
│ 15 │    0 │ Total    │ 18480432 │
│ 16 │    0 │ 0 a 4    │   870693 │
│ 17 │    0 │ 5 a 9    │  1147515 │
│ 18 │    0 │ 85 o más │   275586 │
│ 19 │   15 │ Total    │   244569 │
│ 20 │   15 │ 0 a 4    │    12819 │
│ 21 │   15 │ 5 a 9    │    16414 │
│ 22 │   15 │ 10 a 14  │    18797 │
│ 23 │   15 │ 15 a 19

In [15]:
with open(agedistpath, "r") as f:
    ageDistribution = [r for r in csv.DictReader(f)]

ageDistribution[:3]

[{'code': '0', 'age': '10 a 14', 'total': '1256440'},
 {'code': '0', 'age': '15 a 19', 'total': '1219347'},
 {'code': '0', 'age': '20 a 24', 'total': '1273193'}]

In [16]:
demo.setAgeDistributions(ageDistribution, isFlat=True)

In [17]:
demo.showGeoUnits(max_level=0)

---------------------------------------------------------
GeoUnit
label: Chile
level: 0
code: 0
children: 15 | 1 | 2 | 3 | 4 | 5 | 13 | 6 | 7 | 16 | 8 | 9 | 14 | 10 | 11 | 12
gender distribution: {'female': '9513399', 'male': '8967033', 'total': '18480432'}
age distribution: 
    10 a 14: 1256440
    15 a 19: 1219347
    20 a 24: 1273193
    25 a 29: 1383669
    30 a 34: 1527489
    35 a 39: 1408198
    40 a 44: 1270493
    45 a 49: 1151776
    50 a 54: 1173352
    55 a 59: 1133239
    60 a 64: 1077790
    65 a 69: 870801
    70 a 74: 646241
    75 a 79: 477186
    80 a 84: 317424
    Total: 18480432
    0 a 4: 870693
    5 a 9: 1147515
    85 o más: 275586


##### 3.1 We load first the metadata for users from Chile

#### 3. Users location match

Now that our demograph is created and loaded with demographic data, we will use it to mach the localisations declared by users to the geographical units.

We use the method  **matchAndStoreUsersLocations** from the Demograph.

In [18]:
!xan head data/chile/chile_metadata_2023.csv | xan v 


Displaying 3 cols from 10 rows of <stdin>
┌───┬─────────────────────┬────────────────────────┬─────────────────┐
│ - │ twitter_id          │ location               │ screen_name     │
├───┼─────────────────────┼────────────────────────┼─────────────────┤
│ 0 │  902359033323114496 │ Arica, Chile           │ EnriqueLeeF     │
│ 1 │           369313977 │ Pudahuel (Santiago RM) │ jsotog82        │
│ 2 │           400511087 │ Del Mundo              │ MiaPinUp        │
│ 3 │           605521873 │ Chile                  │ fchhandball     │
│ 4 │            90665520 │ Santiago de Chile      │ dixiatorrejon   │
│ 5 │ 1086025967502528513 │ La Florida, Chile      │ EganaLira       │
│ 6 │          2784569228 │ Toesca 1929, depto 608 │ AndreaPdeLP     │
│ 7 │ 1316901328464023552 │ Angol, Chile           │ ChrisssDavi     │
│ 8 │           120163246 │ Quilpué                │ Cinthya_Carduii │
│ 9 │          2807582413 │ Chillan·               │ leteliercortes  │
└───┴─────────────────────┴───────

In [19]:
with open(usersdatapath, 'r') as f:
    metadata = [r for r in csv.reader(f)]
print(f"Locations file with {len(metadata)} entries loaded from {agedistpath}")
metadata[:3]

Locations file with 994908 entries loaded from data/chile/chile_age_distribution_census_2024.csv


[['twitter_id', 'location', 'screen_name'],
 ['902359033323114496', 'Arica, Chile', 'EnriqueLeeF'],
 ['369313977', 'Pudahuel (Santiago RM)', 'jsotog82']]

In [20]:
match_kwargs = {
    "stopwords": ["el", "la", "lo", "les", "las", "los", "de", "del", "en"],
    "split_characters": ["-", "/", "|", ".", "'", "(", ")"],
    "search_index": [1],
    "has_headers": True,
}

In [21]:
aliases = {demo.countryCode: getCountryAliases(country)}
aliases.update(getUnitsAliases(country))
aliases

{'0': ['chile', 'chili']}

In [22]:
other_countries = getOtherCountriesNames(country)
banned_words = other_countries
list(banned_words)[:5]

['maldives', 'spain', 'norway', 'vanuatu', 'qatar']

In [23]:
start = time.time()
users_matched_locations = demo.matchAndStoreUsersLocations(
    data=metadata,
    aliases=aliases,
    banned_words=banned_words,
    verbose=False,
    **match_kwargs)

duration = time.time() - start
print(f"Whole matching {len(metadata)} users locations took {duration} seconds.")

Whole matching 994908 users locations took 28.276033878326416 seconds.


In [24]:
demo.showGeoUnits(max_level=0)

---------------------------------------------------------
GeoUnit
label: Chile
level: 0
code: 0
children: 15 | 1 | 2 | 3 | 4 | 5 | 13 | 6 | 7 | 16 | 8 | 9 | 14 | 10 | 11 | 12
gender distribution: {'female': '9513399', 'male': '8967033', 'total': '18480432'}
age distribution: 
    10 a 14: 1256440
    15 a 19: 1219347
    20 a 24: 1273193
    25 a 29: 1383669
    30 a 34: 1527489
    35 a 39: 1408198
    40 a 44: 1270493
    45 a 49: 1151776
    50 a 54: 1173352
    55 a 59: 1133239
    60 a 64: 1077790
    65 a 69: 870801
    70 a 74: 646241
    75 a 79: 477186
    80 a 84: 317424
    Total: 18480432
    0 a 4: 870693
    5 a 9: 1147515
    85 o más: 275586
nb localized users: 555278
localized users examples: 
    ['902359033323114496', 'Arica, Chile', 'EnriqueLeeF', 'arica chile']
    ['605521873', 'Chile', 'fchhandball', 'chile']
    ['90665520', 'Santiago de Chile', 'dixiatorrejon', 'santiago chile']
    ['1086025967502528513', 'La Florida, Chile', 'EganaLira', 'florida chile']
    

#### 4. Exports

Now that the demograph has matched and store users is their geographical units, we export flat tables with  ...

In [25]:
excel_export_path = "results/20251201/chile/chile_units_users_reports_nuts_2024_epo_2023.xlsx"
stats_export_pattern = "results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_${level}.csv"
users_export_path = "results/20251201/chile/localized_users_nuts_2024_epo_2023.csv"
full_users_export_path = "results/20251201/chile/localized_users_full_nuts_2024_epo_2023.csv"
units_export_path = "results/20251201/chile/units_nuts_2024.csv"


##### 4.1 Export matchs stats per level for cloropleths visualizations

In [26]:
for level in range(demo.getDeepestLevel() + 1) :
    path = Template(stats_export_pattern).safe_substitute(level=level)
    demo.exportLocalizationsMatchesPerc(level, path, descendants=True, add_headers=False)

File saved as results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_0.csv
File saved as results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_1.csv
File saved as results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_2.csv
File saved as results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_3.csv


In [27]:
!xan v results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_1.csv


Displaying 2 cols from 15 rows of results/20251201/chile/nb_matchs_perc_chile_nuts_2024_epo_2023_level_1.csv
┌────┬────┬────────────────────┐
│ -  │ 15 │ 3.324624134702272  │
├────┼────┼────────────────────┤
│ 0  │  1 │ 3.339588865513269  │
│ 1  │  2 │ 3.3727510796076903 │
│ 2  │  3 │ 2.723778327428304  │
│ 3  │  4 │ 2.653494448073155  │
│ 4  │  5 │ 3.5548056937226966 │
│ 5  │ 13 │ 3.668335373444362  │
│ 6  │  6 │ 2.0382323029735785 │
│ 7  │  7 │ 2.2283901806576623 │
│ 8  │ 16 │ 2.099400924087771  │
│ 9  │  8 │ 3.0150788036891396 │
│ 10 │  9 │ 2.2125387090357207 │
│ 11 │ 14 │ 3.596414132536474  │
│ 12 │ 10 │ 2.928953008253546  │
│ 13 │ 11 │ 3.6408754776912007 │
│ 14 │ 12 │ 4.795931234500442  │
└────┴────┴────────────────────┘



##### 4.2 export localized users

In [28]:
localizedUsers, localizedUsersColumns = demo.exportLocalizedUsers(users_export_path, full_path=full_users_export_path)

Localized users file saved as results/20251201/chile/localized_users_nuts_2024_epo_2023.csv
Localized users file saved as results/20251201/chile/localized_users_full_nuts_2024_epo_2023.csv


In [29]:
! xan head results/20251201/chile/localized_users_nuts_2024_epo_2023.csv | xan v


Displaying 4/12 cols from 10 rows of <stdin>
┌───┬─────────────────────┬─────────────────────┬───┬──────────┬───────────────┐
│ - │ twitter_id          │ location            │ … │ level_3… │ level_3_label │
├───┼─────────────────────┼─────────────────────┼───┼──────────┼───────────────┤
│ 0 │ 1000000362382798848 │ Chili               │ … │  <empty> │ <empty>       │
│ 1 │ 1000000978979053571 │ Molina, Chile       │ … │     7304 │ Molina        │
│ 2 │ 1000001110050996224 │ La Reina, Chile     │ … │    13113 │ La Reina      │
│ 3 │ 1000003557482024961 │ Chile               │ … │  <empty> │ <empty>       │
│ 4 │           100000389 │ Santiago, Chile·    │ … │    13101 │ Santiago      │
│ 5 │ 1000005875937968131 │ Puente Alto, Chile  │ … │    13201 │ Puente Alto   │
│ 6 │ 1000008780363567105 │ Santiago, Chile     │ … │    13101 │ Santiago      │
│ 7 │           100001239 │ Concepción, Chile.  │ … │     8101 │ Concepción    │
│ 8 │           100001345 │ concepcion          │ … │     8101 

In [30]:
! xan head results/20251201/chile/localized_users_full_nuts_2024_epo_2023.csv | xan v


Displaying 4/8 cols from 10 rows of <stdin>
┌───┬─────────────────────┬─────────────────────┬───┬───────────┬──────────────┐
│ - │ twitter_id          │ location            │ … │ level_2_… │ level_3_code │
├───┼─────────────────────┼─────────────────────┼───┼───────────┼──────────────┤
│ 0 │ 1000000362382798848 │ Chili               │ … │ <empty>   │      <empty> │
│ 1 │ 1000000978979053571 │ Molina, Chile       │ … │ p73       │         7304 │
│ 2 │ 1000001110050996224 │ La Reina, Chile     │ … │ p131      │        13113 │
│ 3 │ 1000003557482024961 │ Chile               │ … │ <empty>   │      <empty> │
│ 4 │           100000389 │ Santiago, Chile·    │ … │ p131      │        13101 │
│ 5 │ 1000005875937968131 │ Puente Alto, Chile  │ … │ p132      │        13201 │
│ 6 │ 1000008780363567105 │ Santiago, Chile     │ … │ p131      │        13101 │
│ 7 │           100001239 │ Concepción, Chile.  │ … │ p81       │         8101 │
│ 8 │           100001345 │ concepcion          │ … │ p81       

##### 4.3 export flatten units excel file to monitoring and debugging

In [32]:
import pandas as pd
unitReport, unitsColumns = demo.exportUnitsReport(units_export_path)
with pd.ExcelWriter(excel_export_path) as writer:

    unitsColumns = [" ".join(c.split("_")) for c in unitsColumns]
    pd.DataFrame(data=unitReport, columns=unitsColumns) \
        .to_excel(writer, index=False, sheet_name=f"units stats")

    localizedUsersColumns = [" ".join(c.split("_")) for c in localizedUsersColumns]
    df = pd.DataFrame(data=localizedUsers, columns=localizedUsersColumns)
    df = df.sample(n=min(len(df), 10000), random_state=84)
    try:
        df.to_excel(writer, index=False, sheet_name=f"localized users")
    except:
        df['location'] = df['location'].apply(lambda x: x.encode('unicode_escape').decode('utf-8') if isinstance(x, str) else x)
        df.to_excel(writer, index=False, sheet_name=f"localized users")